In [96]:
# Importing necessary libraries

import pandas as pd
import duckdb 

In [97]:
# Creating connect

con = duckdb.connect("../01_docs/01_data/project.duckdb")

In [ ]:
# Testing connection

con.execute(
    """
    SELECT *
    FROM read_parquet('../01_docs/01_data/01_raw/fhvhv_tripdata_2024-07.parquet')
    LIMIT 10
    """
).df()

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B03404,B03404,2024-07-01 00:13:16,2024-07-01 00:18:28,2024-07-01 00:19:43,2024-07-01 00:40:35,138,141,8.840,...,4.70,2.75,2.5,9.28,24.19,N,N,N,N,N
1,HV0003,B03404,B03404,2024-07-01 00:16:44,2024-07-01 00:18:58,2024-07-01 00:21:00,2024-07-01 00:41:24,61,80,3.960,...,2.22,0.00,0.0,0.00,19.37,N,N,N,N,N
2,HV0003,B03404,B03404,2024-07-01 00:38:38,2024-07-01 00:45:10,2024-07-01 00:45:10,2024-07-01 01:01:05,36,260,5.610,...,1.84,0.00,0.0,0.00,12.08,Y,Y,N,N,N
3,HV0003,B03404,B03404,2024-07-01 00:38:26,2024-07-01 00:48:38,2024-07-01 00:49:01,2024-07-01 01:15:02,80,42,11.240,...,4.05,0.00,0.0,0.00,25.64,Y,Y,N,N,N
4,HV0003,B03404,B03404,2024-07-01 00:34:57,2024-07-01 00:40:58,2024-07-01 00:41:36,2024-07-01 00:49:48,152,41,1.600,...,1.03,0.00,0.0,5.00,7.11,N,N,N,N,N
5,HV0005,B03406,None,2024-07-01 00:36:16,NaT,2024-07-01 00:44:05,2024-07-01 00:54:37,138,92,3.343,...,1.93,0.00,2.5,0.00,11.45,N,N,N,N,N
6,HV0003,B03404,B03404,2024-07-01 00:05:41,2024-07-01 00:09:27,2024-07-01 00:09:49,2024-07-01 00:35:52,164,17,5.520,...,3.27,2.75,0.0,0.00,25.95,N,N,N,N,N
7,HV0003,B03404,B03404,2024-07-01 00:36:11,2024-07-01 00:40:27,2024-07-01 00:41:13,2024-07-01 00:49:39,37,225,1.680,...,1.19,0.00,0.0,0.00,8.19,N,N,N,N,N
8,HV0003,B03404,B03404,2024-07-01 00:48:32,2024-07-01 00:50:15,2024-07-01 00:51:13,2024-07-01 01:21:50,225,79,6.860,...,1.55,0.75,0.0,0.00,16.04,Y,Y,N,N,N
9,HV0003,B03404,B03404,2024-07-01 00:47:59,2024-07-01 00:52:47,2024-07-01 00:53:03,2024-07-01 01:01:39,17,49,1.370,...,0.43,0.00,0.0,0.00,2.95,Y,Y,N,N,N


In [ ]:
# Creating a view

def create_view(con, category):
    con.execute(
        f"""
        CREATE OR REPLACE VIEW {category} AS
        SELECT *
        FROM read_parquet('../01_docs/01_data/01_raw/{category}_tripdata_2024-**.parquet')
        """
        )


create_view(con, 'fhvhv')

In [100]:
# Checking features of the created view

con.execute("DESCRIBE fhvhv").df()

,column_name,column_type,null,key,default,extra
0,hvfhs_license_num,VARCHAR,YES,None,None,None
1,dispatching_base_num,VARCHAR,YES,None,None,None
2,originating_base_num,VARCHAR,YES,None,None,None
3,request_datetime,TIMESTAMP,YES,None,None,None
4,on_scene_datetime,TIMESTAMP,YES,None,None,None
5,pickup_datetime,TIMESTAMP,YES,None,None,None
6,dropoff_datetime,TIMESTAMP,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,trip_miles,DOUBLE,YES,None,None,None


In [101]:
# Creating a view with selected columns and renamed fields

con.execute("""
            CREATE OR REPLACE VIEW fhvhv_selected AS
            SELECT
              hvfhs_license_num     AS hvfhs_license_num,
              pickup_datetime       AS pickup_datetime,
              dropoff_datetime      AS dropoff_datetime,
              request_datetime      AS request_datetime,
              on_scene_datetime     AS on_scene_datetime,
              trip_miles            AS trip_distance,
              trip_time             AS trip_time,
              PULocationID          AS pickup_location,
              DOLocationID          AS dropoff_location,
              base_passenger_fare   AS fare_amount,
              tips                  AS tip_amount,
              tolls                 AS tolls_amount,
              airport_fee           AS airport_fee,
              driver_pay            AS driver_pay,
              shared_request_flag   AS shared_request,
              shared_match_flag     AS shared_match,
              wav_request_flag      AS wav_request,
              wav_match_flag        AS wav_match,
              access_a_ride_flag    AS access_a_ride
            FROM fhvhv
            """)

In [102]:
# Checking for size of the created view

con.execute("SELECT COUNT(*) FROM fhvhv_selected").df()

,count_star()
0,239470448


In [103]:
# Checking features of the created view

con.execute("DESCRIBE fhvhv_selected").df()

,column_name,column_type,null,key,default,extra
0,hvfhs_license_num,VARCHAR,YES,None,None,None
1,pickup_datetime,TIMESTAMP,YES,None,None,None
2,dropoff_datetime,TIMESTAMP,YES,None,None,None
3,request_datetime,TIMESTAMP,YES,None,None,None
4,on_scene_datetime,TIMESTAMP,YES,None,None,None
5,trip_distance,DOUBLE,YES,None,None,None
6,trip_time,BIGINT,YES,None,None,None
7,pickup_location,INTEGER,YES,None,None,None
8,dropoff_location,INTEGER,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


In [104]:
# Checking unique values in categorical columns

In [105]:
con.execute("SELECT COUNT(*) FROM fhvhv_selected GROUP BY hvfhs_license_num").df()

,count_star()
0,179125798
1,60344650


In [106]:
con.execute("SELECT COUNT(*) FROM fhvhv_selected GROUP BY shared_request").df()

,count_star()
0,230799006
1,8671442


In [107]:
con.execute("SELECT COUNT(*) FROM fhvhv_selected GROUP BY shared_match").df()

,count_star()
0,3726194
1,235744254


In [108]:
con.execute("SELECT COUNT(*) FROM fhvhv_selected GROUP BY wav_request").df()

,count_star()
0,238831787
1,638661


In [109]:
con.execute("SELECT COUNT(*) FROM fhvhv_selected GROUP BY wav_match").df()

,count_star()
0,21860980
1,217609468


In [110]:
con.execute("SELECT COUNT(*) FROM fhvhv_selected GROUP BY access_a_ride").df()

,count_star()
0,225398
1,239245050


In [111]:
# Excluding acces a ride flags

con.execute("""
            CREATE OR REPLACE VIEW fhvhv_no_access_a_ride AS
            SELECT *
            FROM fhvhv_selected
            WHERE access_a_ride = 'N'
            """)

In [112]:
# Checking excluded view

con.execute("SELECT COUNT(*) FROM fhvhv_no_access_a_ride GROUP BY access_a_ride").df()

,count_star()
0,239245050


In [113]:
# Creating a view with boolean fields

con.execute("""
            CREATE OR REPLACE VIEW fhvhv_selected_bool AS
            SELECT    
                CASE hvfhs_license_num
                  WHEN 'HV0002' THEN 'Juno'
                  WHEN 'HV0003' THEN 'Uber'
                  WHEN 'HV0004' THEN 'Via'
                  WHEN 'HV0005' THEN 'Lyft'
                  ELSE NULL
                END AS hvfhs_license_num,
            
                pickup_datetime,
                dropoff_datetime,
                request_datetime,
                on_scene_datetime,
                trip_distance,
                trip_time,
                pickup_location,
                dropoff_location,
                fare_amount,
                tip_amount,
                tolls_amount,
                airport_fee,
                driver_pay,
            
                CASE WHEN shared_request = 'Y' THEN TRUE
                  WHEN shared_request = 'N' THEN FALSE
                  ELSE NULL
                END AS shared_request_bool,

                CASE WHEN shared_match = 'Y' THEN TRUE
                  WHEN shared_match = 'N' THEN FALSE
                  ELSE NULL
                END AS shared_match_bool,

                CASE WHEN wav_request = 'Y' THEN TRUE
                  WHEN wav_request = 'N' THEN FALSE
                  ELSE NULL
                END AS wav_request_bool,

                CASE WHEN wav_match = 'Y' THEN TRUE
                  WHEN wav_match = 'N' THEN FALSE
                  ELSE NULL
                END AS wav_match_bool
            FROM fhvhv_no_access_a_ride
            """)

In [114]:
# Checking for missing values in the selected view

column_names = [
    'hvfhs_license_num',
    'pickup_datetime',
    'dropoff_datetime',
    'request_datetime',
    'on_scene_datetime',
    'trip_distance',
    'trip_time',
    'pickup_location',
    'dropoff_location',
    'fare_amount',
    'tip_amount',
    'tolls_amount',
    'airport_fee',
    'driver_pay',
    'shared_request',
    'shared_match',
    'wav_request',
    'wav_match'
]

null_count_expressions = ""
for column_name in column_names:
    null_count_expressions += f"COUNT(*) FILTER (WHERE {column_name} IS NULL) AS {column_name}_null_count,\n"

con.execute(f"""
            CREATE OR REPLACE VIEW null_counts AS
            SELECT
                {null_count_expressions.rstrip(',\n')}
            FROM fhvhv_no_access_a_ride
            """)

In [115]:
con.execute("SELECT * FROM null_counts").df()

# Only missing values are in on_scene_datetime

,hvfhs_license_num_null_count,pickup_datetime_null_count,dropoff_datetime_null_count,request_datetime_null_count,on_scene_datetime_null_count,trip_distance_null_count,trip_time_null_count,pickup_location_null_count,dropoff_location_null_count,fare_amount_null_count,tip_amount_null_count,tolls_amount_null_count,airport_fee_null_count,driver_pay_null_count,shared_request_null_count,shared_match_null_count,wav_request_null_count,wav_match_null_count
0,0,0,0,0,60088004,0,0,0,0,0,0,0,0,0,0,0,0,0


In [116]:
con.execute("SELECT AVG(pickup_datetime - on_scene_datetime) AS avg_time_diff FROM fhvhv_selected WHERE on_scene_datetime IS NOT NULL").df()

# Time difference can be useful for analysis, so I will keep the column

,avg_time_diff
0,0 days 00:01:49.542307


In [117]:
# Checking end view

con.execute("DESCRIBE fhvhv_selected_bool").df()

,column_name,column_type,null,key,default,extra
0,hvfhs_license_num,VARCHAR,YES,None,None,None
1,pickup_datetime,TIMESTAMP,YES,None,None,None
2,dropoff_datetime,TIMESTAMP,YES,None,None,None
3,request_datetime,TIMESTAMP,YES,None,None,None
4,on_scene_datetime,TIMESTAMP,YES,None,None,None
5,trip_distance,DOUBLE,YES,None,None,None
6,trip_time,BIGINT,YES,None,None,None
7,pickup_location,INTEGER,YES,None,None,None
8,dropoff_location,INTEGER,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


In [ ]:
# Saving the DuckDB database

con.execute(
"""
COPY (
  SELECT *
  FROM fhvhv_selected_bool
)
TO '../01_docs/01_data/02_intermediate/fhvhv_clean.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""
)